# Cybersecurity Network Analyzer & Vulnerability Scanner

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:


import subprocess
import sys
import socket
import threading
import hashlib
import requests
import json
import time
import random
import re
from datetime import datetime
import ipaddress
import base64
from urllib.parse import urlparse
import warnings
warnings.filterwarnings('ignore')

# Install required packages for Colab
def install_packages():
    """Install required packages for Colab environment"""
    packages = [
        'python-nmap',
        'requests',
        'scapy',
        'cryptography',
        'python-whois'
    ]

    for package in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"✓ {package} installed successfully")
        except:
            print(f"⚠ Failed to install {package}")

# Run installation
print("Installing required packages...")
install_packages()

import nmap
try:
    import whois
except:
    print("⚠ Whois module not available")

class CyberSecurityAnalyzer:
    def __init__(self):
        self.results = {}
        self.threats_db = self.load_threat_signatures()

    def load_threat_signatures(self):
        """Load common threat signatures and patterns"""
        return {
            'sql_injection': [
                r"(\bunion\b.*\bselect\b)",
                r"(\bselect\b.*\bfrom\b.*\bwhere\b)",
                r"(\bor\b.*=.*)",
                r"(1=1|1='1'|'=')"
            ],
            'xss': [
                r"<script.*?>.*?</script>",
                r"javascript:",
                r"on\w+\s*=",
                r"<iframe.*?>"
            ],
            'suspicious_domains': [
                'bit.ly', 'tinyurl.com', 'goo.gl', 't.co'
            ],
            'malware_indicators': [
                'ransomware', 'trojan', 'backdoor', 'keylogger'
            ]
        }

    def banner(self):
        """Display project banner"""
        print("""
╔══════════════════════════════════════════════════════════════╗
║                 CYBERSECURITY ANALYZER v2.0                 ║
║              Network Security & Vulnerability Scanner        ║
║                     Google Colab Edition                     ║
╚══════════════════════════════════════════════════════════════╝
        """)

    def port_scanner(self, target, ports):
        """Advanced port scanner with service detection"""
        print(f"\n🔍 Scanning {target} for open ports...")

        try:
            nm = nmap.PortScanner()
            result = nm.scan(target, f"{ports[0]}-{ports[-1]}", '-sS -sV -A')

            open_ports = []
            for host in nm.all_hosts():
                for protocol in nm[host].all_protocols():
                    ports_list = nm[host][protocol].keys()
                    for port in ports_list:
                        state = nm[host][protocol][port]['state']
                        if state == 'open':
                            service = nm[host][protocol][port].get('name', 'unknown')
                            version = nm[host][protocol][port].get('version', '')
                            open_ports.append({
                                'port': port,
                                'service': service,
                                'version': version,
                                'state': state
                            })

            return open_ports

        except Exception as e:
            print(f"❌ Nmap scan failed: {e}")
            return self.basic_port_scan(target, ports)

    def basic_port_scan(self, target, ports):
        """Basic port scanner fallback"""
        open_ports = []

        def scan_port(port):
            try:
                sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                sock.settimeout(1)
                result = sock.connect_ex((target, port))
                if result == 0:
                    open_ports.append({
                        'port': port,
                        'service': 'unknown',
                        'version': '',
                        'state': 'open'
                    })
                sock.close()
            except:
                pass

        threads = []
        for port in ports:
            t = threading.Thread(target=scan_port, args=(port,))
            threads.append(t)
            t.start()

        for t in threads:
            t.join()

        return sorted(open_ports, key=lambda x: x['port'])

    def vulnerability_check(self, open_ports, target):
        """Check for common vulnerabilities based on open ports"""
        vulnerabilities = []

        vuln_db = {
            21: "FTP - Check for anonymous login, weak passwords",
            22: "SSH - Check for weak passwords, outdated versions",
            23: "Telnet - Unencrypted protocol, high risk",
            25: "SMTP - Check for open relay, weak authentication",
            53: "DNS - Check for zone transfers, cache poisoning",
            80: "HTTP - Check for web vulnerabilities, outdated software",
            110: "POP3 - Check for weak passwords, encryption",
            143: "IMAP - Check for weak passwords, encryption",
            443: "HTTPS - Check SSL/TLS configuration, certificates",
            993: "IMAPS - Secure IMAP, verify certificate",
            995: "POP3S - Secure POP3, verify certificate",
            3389: "RDP - Check for weak passwords, outdated versions",
            5432: "PostgreSQL - Check for weak passwords, configuration",
            3306: "MySQL - Check for weak passwords, configuration",
            1433: "SQL Server - Check for weak passwords, configuration"
        }

        for port_info in open_ports:
            port = port_info['port']
            if port in vuln_db:
                risk_level = self.assess_risk_level(port, port_info['service'])
                vulnerabilities.append({
                    'port': port,
                    'service': port_info['service'],
                    'description': vuln_db[port],
                    'risk_level': risk_level
                })

        return vulnerabilities

    def assess_risk_level(self, port, service):
        """Assess risk level based on port and service"""
        high_risk_ports = [21, 23, 3389, 1433, 3306, 5432]
        medium_risk_ports = [22, 25, 53, 80, 110, 143]

        if port in high_risk_ports:
            return "HIGH"
        elif port in medium_risk_ports:
            return "MEDIUM"
        else:
            return "LOW"

    def web_vulnerability_scan(self, url):
        """Basic web vulnerability scanner"""
        print(f"\n🌐 Scanning {url} for web vulnerabilities...")

        vulnerabilities = []

        try:
            # Check for common files
            common_files = [
                '/robots.txt', '/sitemap.xml', '/.htaccess',
                '/config.php', '/wp-config.php', '/admin'
            ]

            for file_path in common_files:
                test_url = url.rstrip('/') + file_path
                try:
                    response = requests.get(test_url, timeout=5)
                    if response.status_code == 200:
                        vulnerabilities.append({
                            'type': 'Information Disclosure',
                            'description': f"Accessible file: {file_path}",
                            'risk': 'MEDIUM'
                        })
                except:
                    pass

            # Check HTTP headers
            try:
                response = requests.get(url, timeout=10)
                headers = response.headers

                security_headers = {
                    'X-Frame-Options': 'Clickjacking protection',
                    'X-XSS-Protection': 'XSS protection',
                    'X-Content-Type-Options': 'MIME type sniffing protection',
                    'Strict-Transport-Security': 'HTTP Strict Transport Security'
                }

                for header, description in security_headers.items():
                    if header not in headers:
                        vulnerabilities.append({
                            'type': 'Missing Security Header',
                            'description': f"Missing {header} - {description}",
                            'risk': 'MEDIUM'
                        })

                # Check for server information disclosure
                if 'Server' in headers:
                    vulnerabilities.append({
                        'type': 'Information Disclosure',
                        'description': f"Server banner: {headers['Server']}",
                        'risk': 'LOW'
                    })

            except Exception as e:
                vulnerabilities.append({
                    'type': 'Connection Error',
                    'description': f"Could not connect to {url}: {str(e)}",
                    'risk': 'INFO'
                })

        except Exception as e:
            print(f"❌ Web scan error: {e}")

        return vulnerabilities

    def password_strength_checker(self, password):
        """Analyze password strength"""
        score = 0
        feedback = []

        # Length check
        if len(password) >= 12:
            score += 2
        elif len(password) >= 8:
            score += 1
        else:
            feedback.append("Password should be at least 8 characters long")

        # Character variety checks
        if re.search(r'[a-z]', password):
            score += 1
        else:
            feedback.append("Add lowercase letters")

        if re.search(r'[A-Z]', password):
            score += 1
        else:
            feedback.append("Add uppercase letters")

        if re.search(r'\d', password):
            score += 1
        else:
            feedback.append("Add numbers")

        if re.search(r'[!@#$%^&*()_+\-=\[\]{};:,.<>?]', password):
            score += 1
        else:
            feedback.append("Add special characters")

        # Common patterns check
        common_patterns = ['123', 'abc', 'password', 'qwerty', '111']
        for pattern in common_patterns:
            if pattern.lower() in password.lower():
                score -= 1
                feedback.append(f"Avoid common patterns like '{pattern}'")

        # Strength assessment
        if score >= 6:
            strength = "VERY STRONG"
        elif score >= 4:
            strength = "STRONG"
        elif score >= 2:
            strength = "MEDIUM"
        else:
            strength = "WEAK"

        return {
            'strength': strength,
            'score': max(0, score),
            'feedback': feedback
        }

    def hash_analyzer(self, hash_input):
        """Analyze and identify hash types"""
        hash_types = {
            32: ["MD5", "NTLM"],
            40: ["SHA-1"],
            56: ["SHA-224"],
            64: ["SHA-256"],
            96: ["SHA-384"],
            128: ["SHA-512"]
        }

        length = len(hash_input)
        possible_types = hash_types.get(length, ["Unknown"])

        # Generate sample hashes for comparison
        sample_text = "password123"
        sample_hashes = {
            "MD5": hashlib.md5(sample_text.encode()).hexdigest(),
            "SHA-1": hashlib.sha1(sample_text.encode()).hexdigest(),
            "SHA-256": hashlib.sha256(sample_text.encode()).hexdigest(),
            "SHA-512": hashlib.sha512(sample_text.encode()).hexdigest()
        }

        return {
            'input_hash': hash_input,
            'length': length,
            'possible_types': possible_types,
            'sample_hashes': sample_hashes
        }

    def generate_security_report(self, scan_results):
        """Generate comprehensive security report"""
        report = f"""
╔══════════════════════════════════════════════════════════════╗
║                    SECURITY ASSESSMENT REPORT               ║
║                Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}                ║
╚══════════════════════════════════════════════════════════════╝

TARGET INFORMATION:
└─ Target: {scan_results.get('target', 'N/A')}
└─ Scan Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

OPEN PORTS SUMMARY:
"""

        if 'open_ports' in scan_results:
            for port in scan_results['open_ports']:
                report += f"└─ Port {port['port']} ({port['service']}) - {port['state'].upper()}\n"

        report += "\nVULNERABILITY ASSESSMENT:\n"

        if 'vulnerabilities' in scan_results:
            high_count = medium_count = low_count = 0
            for vuln in scan_results['vulnerabilities']:
                risk = vuln.get('risk_level', vuln.get('risk', 'UNKNOWN'))
                if risk == 'HIGH':
                    high_count += 1
                elif risk == 'MEDIUM':
                    medium_count += 1
                else:
                    low_count += 1

                report += f"└─ [{risk}] {vuln.get('description', vuln.get('type', 'Unknown'))}\n"

            report += f"\nRISK SUMMARY:\n"
            report += f"└─ High Risk: {high_count}\n"
            report += f"└─ Medium Risk: {medium_count}\n"
            report += f"└─ Low Risk: {low_count}\n"

        report += "\nRECOMMENDATIONS:\n"
        report += "└─ Close unnecessary ports\n"
        report += "└─ Update software and services regularly\n"
        report += "└─ Implement strong authentication\n"
        report += "└─ Use encryption for sensitive data\n"
        report += "└─ Regular security monitoring\n"

        return report

# Main execution
def main():
    analyzer = CyberSecurityAnalyzer()
    analyzer.banner()

    while True:
        print("\n" + "="*60)
        print("CYBERSECURITY ANALYZER MENU")
        print("="*60)
        print("1. 🔍 Port Scanner")
        print("2. 🌐 Web Vulnerability Scanner")
        print("3. 🔐 Password Strength Checker")
        print("4. 🧬 Hash Analyzer")
        print("5. 📊 Generate Demo Security Report")
        print("6. 🎯 Full Security Assessment")
        print("7. ❌ Exit")

        choice = input("\nSelect an option (1-7): ").strip()

        if choice == '1':
            target = input("Enter target IP/hostname (or press Enter for demo): ").strip()
            if not target:
                target = "scanme.nmap.org"
                print(f"Using demo target: {target}")

            ports_input = input("Enter port range (e.g., 80,443 or 1-1000) or press Enter for common ports: ").strip()
            if not ports_input:
                ports = [21, 22, 23, 25, 53, 80, 110, 143, 443, 993, 995, 3389]
            else:
                try:
                    if '-' in ports_input:
                        start, end = map(int, ports_input.split('-'))
                        ports = list(range(start, end + 1))
                    else:
                        ports = [int(p.strip()) for p in ports_input.split(',')]
                except:
                    ports = [80, 443, 22]
                    print("Invalid port format, using default ports")

            open_ports = analyzer.port_scanner(target, ports)

            if open_ports:
                print(f"\n✅ Found {len(open_ports)} open ports:")
                for port in open_ports:
                    print(f"  └─ Port {port['port']}: {port['service']} ({port['state']})")

                # Check vulnerabilities
                vulns = analyzer.vulnerability_check(open_ports, target)
                if vulns:
                    print(f"\n⚠️  Potential vulnerabilities found:")
                    for vuln in vulns:
                        print(f"  └─ [{vuln['risk_level']}] Port {vuln['port']}: {vuln['description']}")
            else:
                print("❌ No open ports found")

        elif choice == '2':
            url = input("Enter URL to scan (or press Enter for demo): ").strip()
            if not url:
                url = "http://httpbin.org"
                print(f"Using demo URL: {url}")

            if not url.startswith(('http://', 'https://')):
                url = 'http://' + url

            vulns = analyzer.web_vulnerability_scan(url)

            if vulns:
                print(f"\n⚠️  Web vulnerabilities found:")
                for vuln in vulns:
                    print(f"  └─ [{vuln['risk']}] {vuln['type']}: {vuln['description']}")
            else:
                print("✅ No obvious web vulnerabilities detected")

        elif choice == '3':
            password = input("Enter password to analyze: ").strip()
            if not password:
                password = "Password123!"
                print(f"Using demo password: {password}")

            result = analyzer.password_strength_checker(password)

            print(f"\n🔐 Password Analysis Results:")
            print(f"  └─ Strength: {result['strength']}")
            print(f"  └─ Score: {result['score']}/6")

            if result['feedback']:
                print(f"  └─ Recommendations:")
                for feedback in result['feedback']:
                    print(f"     • {feedback}")

        elif choice == '4':
            hash_input = input("Enter hash to analyze: ").strip()
            if not hash_input:
                hash_input = hashlib.md5("password123".encode()).hexdigest()
                print(f"Using demo MD5 hash: {hash_input}")

            result = analyzer.hash_analyzer(hash_input)

            print(f"\n🧬 Hash Analysis Results:")
            print(f"  └─ Hash: {result['input_hash']}")
            print(f"  └─ Length: {result['length']} characters")
            print(f"  └─ Possible types: {', '.join(result['possible_types'])}")

            print(f"\n📝 Sample hashes for 'password123':")
            for hash_type, sample_hash in result['sample_hashes'].items():
                print(f"  └─ {hash_type}: {sample_hash}")

        elif choice == '5':
            # Generate demo report
            demo_results = {
                'target': 'demo.example.com',
                'open_ports': [
                    {'port': 22, 'service': 'ssh', 'state': 'open'},
                    {'port': 80, 'service': 'http', 'state': 'open'},
                    {'port': 443, 'service': 'https', 'state': 'open'}
                ],
                'vulnerabilities': [
                    {'risk_level': 'HIGH', 'description': 'SSH weak password policy detected'},
                    {'risk_level': 'MEDIUM', 'description': 'Missing security headers on web server'},
                    {'risk_level': 'LOW', 'description': 'Server banner information disclosure'}
                ]
            }

            report = analyzer.generate_security_report(demo_results)
            print(report)

        elif choice == '6':
            target = input("Enter target for full assessment (or press Enter for demo): ").strip()
            if not target:
                target = "scanme.nmap.org"
                print(f"Using demo target: {target}")

            print("\n🚀 Starting comprehensive security assessment...")

            # Port scan
            common_ports = [21, 22, 23, 25, 53, 80, 110, 143, 443, 993, 995, 3389]
            open_ports = analyzer.port_scanner(target, common_ports)

            # Vulnerability assessment
            vulnerabilities = analyzer.vulnerability_check(open_ports, target)

            # Web scan if HTTP/HTTPS ports are open
            web_vulns = []
            for port in open_ports:
                if port['port'] in [80, 443, 8080, 8443]:
                    protocol = 'https' if port['port'] in [443, 8443] else 'http'
                    url = f"{protocol}://{target}:{port['port']}"
                    web_vulns.extend(analyzer.web_vulnerability_scan(url))

            # Compile results
            full_results = {
                'target': target,
                'open_ports': open_ports,
                'vulnerabilities': vulnerabilities + web_vulns
            }

            # Generate and display report
            report = analyzer.generate_security_report(full_results)
            print(report)

        elif choice == '7':
            print("\n👋 Thanks for using Cybersecurity Analyzer!")
            break

        else:
            print("❌ Invalid choice. Please select 1-7.")

if __name__ == "__main__":
    main()